# Train PatchTST + CVAE on Colab

Run this notebook's kernel connected to a Colab runtime (VS Code: kernel picker top-right -> "Select Another Kernel" -> Google Colab -> pick a GPU runtime).

Run the cells top to bottom. The clone step is idempotent (pulls if already cloned).

In [1]:
!nvidia-smi --query-gpu=name,memory.total --format=csv

name, memory.total [MiB]
NVIDIA A100-SXM4-40GB, 40960 MiB


In [2]:
import os

REPO_URL = "https://github.com/WoodyChang21/ECE1508_GenAI.git"
BRANCH = "steven"

if not os.path.isdir("ECE1508_GenAI"):
    !git clone -b {BRANCH} {REPO_URL}
else:
    !cd ECE1508_GenAI && git pull

%cd ECE1508_GenAI

remote: Enumerating objects: 9, done.
remote: Counting objects: 100% (9/9), done.
remote: Compressing objects: 100% (2/2), done.
remote: Total 9 (delta 7), reused 9 (delta 7), pack-reused 0 (from 0)
Unpacking objects: 100% (9/9), 4.20 KiB | 1.40 MiB/s, done.
From https://github.com/WoodyChang21/ECE1508_GenAI
   0c1a9ba..2a39639  steven     -> origin/steven
Updating 0c1a9ba..2a39639
Fast-forward
 steven/src/data_pipeline.py          |   9 +-
 steven/src/evaluate.py               | 164 ++++++++++++++++++++++++-----------
 steven/src/models/cvae_inpainting.py |  13 +--
 steven/src/models/patchtst.py        |  13 +--
 4 files changed, 134 insertions(+), 65 deletions(-)
/content/ECE1508_GenAI


In [3]:
# torch is preinstalled on Colab; just need mplfinance + pyyaml
!pip install -q mplfinance pyyaml

In [4]:
import torch
print("CUDA available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))

CUDA available: True
GPU: NVIDIA A100-SXM4-40GB


## Sanity checks (data pipeline tests)

Cheap to run first -- confirms the feature/window logic before committing to a long training run.

In [5]:
!pip install -q pytest
!python -m pytest steven/tests/ -v

============================= test session starts ==============================
platform linux -- Python 3.12.13, pytest-8.4.2, pluggy-1.6.0 -- /usr/bin/python3
cachedir: .pytest_cache
rootdir: /content/ECE1508_GenAI
plugins: typeguard-4.5.2, anyio-4.14.2, langsmith-0.10.2
collected 20 items                                                             

steven/tests/test_data_pipeline.py::test_reconstruct_prices_round_trip PASSED [  5%]
steven/tests/test_data_pipeline.py::test_anchor_correction_matches_close_0_for_all_horizon_bars PASSED [ 10%]
steven/tests/test_data_pipeline.py::test_wick_components_non_negative PASSED [ 15%]
steven/tests/test_data_pipeline.py::test_build_window_shapes_and_masks PASSED [ 20%]
steven/tests/test_data_pipeline.py::test_to_patchtst_input_patch_padding_mask PASSED [ 25%]
steven/tests/test_data_pipeline.py::test_window_sampler_unique_and_within_bounds PASSED [ 30%]
steven/tests/test_data_pipeline.py::test_window_sampler_respects_split_boundary PASSED [ 35%]

## Train PatchTST (real defaults: 20k windows/epoch, 20 epochs, configs/patchtst.yaml)

Drop `--max-epochs`/`--train-windows-per-epoch` overrides below if you want a quick smoke run first instead of the full config.

In [6]:
!python steven/src/train_patchtst.py --config steven/configs/patchtst.yaml --device auto

20:12:27 device: cuda
20:12:28 Gap report: 145 fully missing weekdays, 39 short sessions (<7 bars)
20:12:28 Dropping first row (2010-01-04 09:30:00): no previous close to compute a return from
20:12:31 epoch 1/20  train_loss=0.21474  val_loss=0.10827  (2.4s)
20:12:31   -> saved best checkpoint (val_loss=0.10827) to steven/outputs/patchtst_checkpoint.pt
20:12:33 epoch 2/20  train_loss=0.16254  val_loss=0.09941  (1.9s)
20:12:33   -> saved best checkpoint (val_loss=0.09941) to steven/outputs/patchtst_checkpoint.pt
20:12:35 epoch 3/20  train_loss=0.15028  val_loss=0.09674  (1.9s)
20:12:35   -> saved best checkpoint (val_loss=0.09674) to steven/outputs/patchtst_checkpoint.pt
20:12:37 epoch 4/20  train_loss=0.14166  val_loss=0.10345  (2.0s)
20:12:39 epoch 5/20  train_loss=0.13959  val_loss=0.09764  (2.0s)
20:12:41 epoch 6/20  train_loss=0.13707  val_loss=0.10450  (1.9s)
20:12:43 epoch 7/20  train_loss=0.13825  val_loss=0.09583  (2.0s)
20:12:43   -> saved best checkpoint (val_loss=0.09583) to

## Train CVAE (real defaults: 20k windows/epoch, 30 epochs, configs/cvae.yaml)

In [7]:
!python steven/src/train_cvae.py --config steven/configs/cvae.yaml --device auto

20:13:11 device: cuda
20:13:11 Gap report: 145 fully missing weekdays, 39 short sessions (<7 bars)
20:13:11 Dropping first row (2010-01-04 09:30:00): no previous close to compute a return from
20:13:14 epoch 1/30  beta=0.20  train_loss=0.61511 (kl=0.8045)  val_loss=0.38935 (kl=0.8000)  (2.1s)
20:13:14   -> saved best checkpoint (val_loss=0.38935) to steven/outputs/cvae_checkpoint.pt
20:13:16 epoch 2/30  beta=0.40  train_loss=0.61710 (kl=0.8002)  val_loss=0.51061 (kl=0.8000)  (1.4s)
20:13:17 epoch 3/30  beta=0.60  train_loss=0.75682 (kl=0.8000)  val_loss=0.65035 (kl=0.8000)  (1.4s)
20:13:18 epoch 4/30  beta=0.80  train_loss=0.88392 (kl=0.8013)  val_loss=0.79578 (kl=0.8000)  (1.4s)
20:13:20 epoch 5/30  beta=1.00  train_loss=0.99932 (kl=0.8003)  val_loss=0.95068 (kl=0.8008)  (1.4s)
20:13:21 epoch 6/30  beta=1.00  train_loss=0.97463 (kl=0.8012)  val_loss=0.92293 (kl=0.8009)  (1.4s)
20:13:23 epoch 7/30  beta=1.00  train_loss=0.96526 (kl=0.8018)  val_loss=0.93120 (kl=0.8022)  (1.4s)
20:13:24

## Evaluate both models on the fixed test set

In [8]:
!python steven/src/evaluate.py \
  --patchtst-checkpoint steven/outputs/patchtst_checkpoint.pt \
  --cvae-checkpoint steven/outputs/cvae_checkpoint.pt \
  --device auto

20:13:58 device: cuda
20:13:59 Gap report: 145 fully missing weekdays, 39 short sessions (<7 bars)
20:13:59 Dropping first row (2010-01-04 09:30:00): no previous close to compute a return from
20:13:59 evaluating on 3000 fixed test windows
20:13:59 wrote metrics to steven/outputs/metrics.json
20:13:59 overall: {
  "n_windows": 3000,
  "patchtst_reparam_mae_rmse": [
    0.09788621217012405,
    0.3754692077636719
  ],
  "cvae_reparam_mae_rmse": [
    0.16863998770713806,
    0.4797087013721466
  ],
  "patchtst_ohlc_mae_rmse": [
    2.3864293067098576,
    3.5654908716475995
  ],
  "cvae_ohlc_mae_rmse": [
    17.622737074701007,
    22.800479233709417
  ],
  "patchtst_volume_mae_rmse": [
    1988452.625,
    3505740.0
  ],
  "cvae_volume_mae_rmse": [
    3021506.25,
    4476819.5
  ],
  "patchtst_directional_accuracy": [
    0.5146666666666667,
    0.5313333333333333,
    0.518
  ],
  "cvae_directional_accuracy": [
    0.4666666666666667,
    0.53,
    0.49866666666666665
  ],
  "cvae_av

## Pull results back down

Zips `steven/outputs/` (checkpoints, metrics.json, sample_plots) and downloads it -- or just `git add`/`commit`/`push` from here if you'd rather sync back through the repo.

In [9]:
!zip -r outputs.zip steven/outputs

try:
    from google.colab import files
    files.download("outputs.zip")
except ImportError:
    print("Not in a Colab frontend session -- outputs.zip is in the working dir, grab it manually.")

updating: steven/outputs/ (stored 0%)
updating: steven/outputs/sample_plots/ (stored 0%)
updating: steven/outputs/sample_plots/long_start26495_ctx56.png (deflated 10%)
updating: steven/outputs/sample_plots/short_start26707_ctx14.png (deflated 6%)
updating: steven/outputs/sample_plots/long_start26193_ctx56.png (deflated 5%)
updating: steven/outputs/sample_plots/medium_start24658_ctx35.png (deflated 9%)
updating: steven/outputs/sample_plots/long_start26847_ctx56.png (deflated 11%)
updating: steven/outputs/sample_plots/long_start25774_ctx56.png (deflated 6%)
updating: steven/outputs/sample_plots/medium_start26039_ctx35.png (deflated 6%)
updating: steven/outputs/sample_plots/short_start26791_ctx14.png (deflated 9%)
updating: steven/outputs/sample_plots/short_start25539_ctx14.png (deflated 6%)
updating: steven/outputs/sample_plots/medium_start26219_ctx35.png (deflated 10%)
updating: steven/outputs/sample_plots/short_start25312_ctx14.png (deflated 10%)
updating: steven/outputs/sample_plots/m

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>